In [2]:
import pandas as pd

# 读取三个文件
df_map = pd.read_csv('/home/user/prognosis_lst/evaluate_model/plot/KM曲线/lung1/test_val_time_event.csv')  # 包含 imgPath, Survival.time, deadstatus.event
df_group = pd.read_csv('/home/user/prognosis_lst/evaluate_model/plot/KM曲线/lung1/test_lung1_KM_group.csv')         # 包含 pathmgroup


# 第一步：新加一列时间和event
path_to_time = dict(zip(df_map['imgPath'], df_map['Survival.time']))
df_group['pfs'] = df_group['path'].map(path_to_time)

path_to_event = dict(zip(df_map['imgPath'], df_map['deadstatus.event']))
df_group['ifprogress'] = df_group['path'].map(path_to_event)

df_group

,path,Risk_Group,pfs,ifprogress
0,/storedata1/proccessed/NSCLC/NSCLC_Radiomics_L...,Low Risk,1067,1
1,/storedata1/proccessed/NSCLC/NSCLC_Radiomics_L...,High Risk,60,1
2,/storedata1/proccessed/NSCLC/NSCLC_Radiomics_L...,Low Risk,706,1
3,/storedata1/proccessed/NSCLC/NSCLC_Radiomics_L...,High Risk,2129,1
4,/storedata1/proccessed/NSCLC/NSCLC_Radiomics_L...,Low Risk,2772,0
...,...,...,...,...
65,/storedata1/proccessed/NSCLC/NSCLC_Radiomics_L...,High Risk,583,1
66,/storedata1/proccessed/NSCLC/NSCLC_Radiomics_L...,Low Risk,673,1
67,/storedata1/proccessed/NSCLC/NSCLC_Radiomics_L...,Low Risk,292,1
68,/storedata1/proccessed/NSCLC/NSCLC_Radiomics_L...,Low Risk,1883,1


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
import os

def plot_km_survival(df):
    T = df['pfs']
    E = df['ifprogress']
    groups = df['Risk_Group']
    
    plt.figure(figsize=(9, 6), dpi=300)
    ax = plt.gca() 
    kmf = KaplanMeierFitter()
    
    group_configs = [
        {'name': 'Low Risk', 'color': '#2878b5'}, 
        {'name': 'High Risk', 'color': '#c82423'}
    ]
    
    for config in group_configs:
        mask = (groups == config['name'])
        if mask.any():
            # 1. 拟合
            kmf.fit(T[mask], event_observed=E[mask], label=config['name'])
            
            # 2. 绘制曲线（关闭 show_censored 避免报错）
            kmf.plot_survival_function(
                ax=ax, 
                color=config['color'], 
                lw=2.5, 
                # show_censored=False, # <--- 彻底关闭内置绘制，解决报错
                ci_show=True, 
                ci_alpha=0.15
            )
            
            # 3. 手动绘制删失点 (Censored Data)
            # 筛选出该组中 Event=0 的点
            censored_mask = mask & (E == 0)
            if censored_mask.any():
                # 获取曲线上的生存概率值
                surv_func = kmf.survival_function_
                censored_times = T[censored_mask]
                # 在对应的时间点找到生存率并打上 '+' 号
                for t in censored_times:
                    # 找到最接近 t 的索引值
                    prob = surv_func.loc[:t].iloc[-1, 0]
                    ax.scatter(t, prob, marker='+', color=config['color'], s=60, linewidths=1, zorder=3)

    # 5. 计算 Log-rank Test
    low_mask = (groups == 'Low Risk')
    high_mask = (groups == 'High Risk')
    if low_mask.any() and high_mask.any():
        results = logrank_test(T[low_mask], T[high_mask], 
                               event_observed_A=E[low_mask], event_observed_B=E[high_mask])
        p_value = results.p_value
        p_str = f"P < 0.0001" if p_value < 0.0001 else f"P = {p_value:.4f}"
        ax.text(0.05, 0.1, f'Log-rank {p_str}', transform=ax.transAxes, 
                fontsize=12, fontweight='bold', bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))

    # 6. 图表美化
    plt.title('Kaplan-Meier Survival Estimates', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Time (Days)', fontsize=13)
    plt.ylabel('Survival Probability', fontsize=13)
    plt.ylim(0, 1.05)
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.legend(frameon=False, fontsize=12)
    
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    
    plt.tight_layout()
    save_path = "/home/user/prognosis_lst/evaluate_model/plot/KM曲线/lung1/KM_Survival_Curve_Fixed_临床.png"
    plt.savefig(save_path, dpi=600,bbox_inches='tight')
    plt.show()
    print(f"✅ 修复版 KM 曲线已生成！\n路径: {os.path.abspath(save_path)}")

if __name__ == "__main__":
    plot_km_survival(df_group)